# Testing a line-number vector plus a phase-dependent 2D component

This notebook tests the decomposition

\[h_{i,t} pprox \mu + v\,k_{i,t} + W z(u_{i,t})\]

where `k` is the current line number, `u = character_count / line_length` is within-line phase, `v` is a single line-number direction, and `W z(u)` is an approximately two-dimensional phase trajectory.

The tests are performed in the full activation space before the final attention layer, using held-out examples.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import SplineTransformer

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
N_TRAIN_EXAMPLES = 2_000
N_EVAL_EXAMPLES = 500
COLLECTION_BATCH_SIZE = 128
TRAIN_SEED = 25_001
EVAL_SEED = 25_002
N_SPLINE_KNOTS = 12
RIDGE_ALPHA = 1.0


In [ ]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()

final_layer = cfg["model"]["n_layers"] - 1
ACTIVATION_SITE = f"blocks.{final_layer}.hook_resid_pre"
seq_len = cfg["task"]["seq_len"]
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Activation site: {ACTIVATION_SITE}")


## Collect activations and line-state variables

In [ ]:
def state_variables(token_rows, line_lengths):
    all_counts, all_line_indices = [], []
    for row, width in zip(token_rows.detach().cpu().tolist(), line_lengths.detach().cpu().tolist()):
        running_count = 0
        completed_lines = 0
        row_counts, row_lines = [], []
        for token_id in row:
            row_counts.append(running_count if token_id == vocab.newline_id else running_count + project_data.extract_character_count(vocab.decode_token(token_id)))
            row_lines.append(completed_lines)
            if token_id == vocab.newline_id:
                running_count = 0
                completed_lines += 1
            else:
                running_count += project_data.extract_character_count(vocab.decode_token(token_id))
        all_counts.append(row_counts)
        all_line_indices.append(row_lines)
    return np.asarray(all_counts, dtype=np.float32), np.asarray(all_line_indices, dtype=np.float32)

@torch.no_grad()
def collect_examples(n_examples, seed):
    activation_chunks, count_chunks, line_chunks, width_chunks = [], [], [], []
    collected = 0
    batch_number = 0
    while collected < n_examples:
        size = min(COLLECTION_BATCH_SIZE, n_examples - collected)
        batch = project_data.make_batch(size, vocab, cfg["task"], device,
                                        seed=seed + batch_number * COLLECTION_BATCH_SIZE)
        _, cache = model.run_with_cache(batch.tokens, prepend_bos=False,
                                        names_filter=lambda name: name == ACTIVATION_SITE)
        counts, line_indices = state_variables(batch.tokens, batch.line_lengths)
        activation_chunks.append(cache[ACTIVATION_SITE].detach().cpu())
        count_chunks.append(counts)
        line_chunks.append(line_indices)
        width_chunks.append(batch.line_lengths.detach().cpu().numpy().astype(np.float32))
        collected += size
        batch_number += 1
    return (
        torch.cat(activation_chunks).numpy()[:n_examples],
        np.concatenate(count_chunks)[:n_examples],
        np.concatenate(line_chunks)[:n_examples],
        np.concatenate(width_chunks)[:n_examples],
    )

X_train, counts_train, lines_train, widths_train = collect_examples(N_TRAIN_EXAMPLES, TRAIN_SEED)
X_eval, counts_eval, lines_eval, widths_eval = collect_examples(N_EVAL_EXAMPLES, EVAL_SEED)
d_model = X_train.shape[-1]
print("Train/eval activation shapes:", X_train.shape, X_eval.shape)
print("Observed line-index range:", lines_train.min(), "to", lines_train.max())


## Build nested full-space trajectory models

In [ ]:
width_mean = widths_train.mean()
width_std = max(widths_train.std(), 1.0)
max_line_index = int(max(lines_train.max(), lines_eval.max()))
phase_spline = SplineTransformer(n_knots=N_SPLINE_KNOTS, degree=3, include_bias=False)
phase_spline.fit(np.linspace(0.0, 1.5, 500)[:, None])
n_phase_features = phase_spline.n_features_out_

def phase_features(counts, widths):
    phase = np.clip(counts / widths[:, None], 0.0, 1.5)
    return phase_spline.transform(phase.reshape(-1, 1))

def design_matrix(counts, lines, widths, mode):
    n_examples, n_positions = counts.shape
    phase = phase_features(counts, widths)
    line_scaled = ((lines - lines_train.mean()) / max(lines_train.std(), 1.0)).reshape(-1, 1)
    line_scaled = np.repeat(line_scaled, n_positions, axis=0)
    phase_line = phase * line_scaled
    if mode == "phase":
        return phase
    if mode == "phase_plus_line":
        return np.concatenate([phase, line_scaled], axis=1)
    if mode == "phase_plus_line_onehot":
        onehot = np.eye(max_line_index + 1, dtype=np.float32)[lines.astype(int).reshape(-1)]
        return np.concatenate([phase, onehot], axis=1)
    if mode == "phase_line_interaction":
        return np.concatenate([phase, phase_line, line_scaled], axis=1)
    raise ValueError(mode)

X_train_flat = X_train.reshape(-1, d_model)
X_eval_flat = X_eval.reshape(-1, d_model)
model_specs = {
    "phase": "phase only",
    "phase_plus_line": "phase + one line-number vector",
    "phase_plus_line_onehot": "phase + separate line offsets",
    "phase_line_interaction": "phase + line/phase interaction",
}
models, predictions = {}, {}
for mode, label in model_specs.items():
    F_train = design_matrix(counts_train, lines_train, widths_train, mode)
    F_eval = design_matrix(counts_eval, lines_eval, widths_eval, mode)
    fitted = Ridge(alpha=RIDGE_ALPHA).fit(F_train, X_train_flat)
    models[mode] = fitted
    predictions[mode] = fitted.predict(F_eval).reshape(N_EVAL_EXAMPLES, seq_len, d_model)
    print(label, "features:", F_train.shape[1])


## Held-out comparison of nested models

In [ ]:
rows = []
for mode, label in model_specs.items():
    predicted = predictions[mode]
    rows.append({
        "model": label,
        "MSE": np.mean((X_eval - predicted) ** 2),
        "weighted_R2": r2_score(X_eval_flat, predicted.reshape(-1, d_model), multioutput="variance_weighted"),
    })
mean_baseline = X_train_flat.mean(axis=0)
rows.insert(0, {
    "model": "global mean",
    "MSE": np.mean((X_eval_flat - mean_baseline) ** 2),
    "weighted_R2": r2_score(X_eval_flat, np.broadcast_to(mean_baseline, X_eval_flat.shape), multioutput="variance_weighted"),
})
model_results = pd.DataFrame(rows)
display(model_results.style.format({"MSE": "{:.6f}", "weighted_R2": "{:.4f}"}))


## Is the line-number effect approximately one-dimensional?

In [ ]:
line_model = models["phase_plus_line"]
line_feature_index = n_phase_features
line_vector = line_model.coef_[:, line_feature_index]
line_vector_norm = np.linalg.norm(line_vector)

onehot_model = models["phase_plus_line_onehot"]
line_offset_matrix = onehot_model.coef_[:, n_phase_features:]
line_singular_values = np.linalg.svd(line_offset_matrix.T, compute_uv=False)
line_explained = line_singular_values ** 2 / np.sum(line_singular_values ** 2)

print(f"Single line-number vector norm: {line_vector_norm:.6f}")
print("Line-offset singular-value variance fractions:", line_explained[:6])
print(f"Variance captured by first line-offset direction: {line_explained[0]:.2%}")


## Is the phase component effectively two-dimensional and orthogonal?

In [ ]:
# Use the phase contribution learned by the additive phase + line model.
phase_coef = line_model.coef_[:, :n_phase_features]
phase_grid = np.linspace(0.0, 1.5, 200)
phase_grid_features = phase_spline.transform(phase_grid[:, None])
phase_curve = phase_grid_features @ phase_coef.T
phase_curve -= phase_curve.mean(axis=0, keepdims=True)
phase_u, phase_singular_values, phase_vt = np.linalg.svd(phase_curve, full_matrices=False)
phase_explained = phase_singular_values ** 2 / np.sum(phase_singular_values ** 2)
phase_basis = phase_vt[:2].T

line_unit = line_vector / max(line_vector_norm, 1e-12)
orthogonality = np.abs(phase_basis.T @ line_unit)
print("Phase singular-value variance fractions:", phase_explained[:6])
print(f"Phase variance captured by first two directions: {phase_explained[:2].sum():.2%}")
print("Absolute cosine with line vector for phase directions:", orthogonality)

phase_coordinates = phase_curve @ phase_basis
phase_radius = np.linalg.norm(phase_coordinates, axis=1)
phase_angle = np.unwrap(np.arctan2(phase_coordinates[:, 1], phase_coordinates[:, 0]))
phase_regression = LinearRegression().fit(phase_grid[:, None], phase_radius)
angle_regression = LinearRegression().fit(phase_grid[:, None], phase_angle)
print(f"Linear radius-vs-phase R²: {phase_regression.score(phase_grid[:, None], phase_radius):.4f}")
print(f"Linear angle-vs-phase R²: {angle_regression.score(phase_grid[:, None], phase_angle):.4f}")


## Phase curve in its estimated 2D subspace

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=phase_coordinates[:, 0], y=phase_coordinates[:, 1], mode="lines+markers",
    marker=dict(size=5, color=phase_grid, colorscale="Viridis", colorbar=dict(title="Phase")),
    line=dict(color="royalblue", width=3),
    text=[f"phase={value:.3f}" for value in phase_grid],
    hovertemplate="%{text}<extra>Phase curve</extra>",
))
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", marker=dict(size=8, color="black"), name="Origin"))
fig.update_layout(
    title="Estimated phase trajectory after removing its mean",
    xaxis_title="Phase subspace direction 1", yaxis_title="Phase subspace direction 2",
    height=700,
)
display(fig)


## Held-out residuals after removing the additive line effect

In [ ]:
# Compare held-out residuals from the additive model with and without line number.
phase_pred = predictions["phase"]
line_pred = predictions["phase_plus_line"]
phase_mse = np.mean((X_eval - phase_pred) ** 2)
line_mse = np.mean((X_eval - line_pred) ** 2)
print(f"Held-out MSE, phase only: {phase_mse:.6f}")
print(f"Held-out MSE, phase + line vector: {line_mse:.6f}")
print(f"Additional MSE reduction from line vector: {1 - line_mse / phase_mse:.2%}")
